# Entity Processor
> 
Pipeline:
- **add** raw JSON (or Entity) → compose context  
- **expand** via ContextProcessor  
- **ingest** into GraphManager  
- **index** in a trivial dict for fast lookup

In [ ]:
#| default_exp core.processor

In [ ]:
#| export
from __future__ import annotations
from typing import Dict, Any

from cogitarelink.core.debug import get_logger
from cogitarelink.core.context import ContextProcessor
from cogitarelink.core.entity import Entity
from cogitarelink.core.graph import GraphManager

log = get_logger("processor")

## processor

In [ ]:
#| export
class EntityProcessor:
    def __init__(self,
                 ctx_proc: ContextProcessor | None = None,
                 graph: GraphManager | None = None):
        self.ctx   = ctx_proc or ContextProcessor()
        self.graph = graph or GraphManager()
        self.index: Dict[str, Entity] = {}

    # ------------------------------------------------------------------
    def add(self, data: Dict[str, Any], vocab=["schema"]) -> Entity:
        "Create Entity, expand + ingest, index by sha."
        ent = Entity(vocab=vocab, content=data)
        expanded = self.ctx.expand(ent.as_json)
        nquads   = self.ctx.normalize(expanded)
        self.graph.ingest_nquads(nquads)
        self.index[ent.sha256] = ent
        return ent

## quick test

In [ ]:
#| hide
from fastcore.test import test_eq, test_ne
ep = EntityProcessor()
e  = ep.add({"name": "Alan"})
test_eq(ep.index[e.sha256].content["name"], "Alan")
test_ne(ep.graph.size(), 0)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()